In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, regularizers, initializers, models, callbacks

# ----------------------------
# User-configurable settings
# ----------------------------
data_path = "atp_basic_dataset.npz"  # current file
FEATURE_COUNT = 11                    # input layer neuron count (number of features you want to use)
LEARNING_RATE = 1e-3
EPOCHS = 1000
BATCH_SIZE = 32
L2_LAMBDA = 1e-4
OUT_DIR = Path("results")
OUT_DIR.mkdir(parents=True, exist_ok=True)
cost_plot_path = OUT_DIR / "cost_reduction.png"
hist_out_path = OUT_DIR / "confidence_histogram.png"
SEED = 42

# Reproducibility
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ----------------------------
# Data loading
# ----------------------------
if not os.path.exists(data_path):
    raise FileNotFoundError(f"{data_path} not found!")

data = np.load(data_path, allow_pickle=True)

# Transpose operation (compatible with user's code)
X_train = data['X_train'].T.astype(np.float32)
Y_train = data['Y_train'].T.astype(np.float32).reshape(-1, 1)
X_val   = data['X_val'].T.astype(np.float32)
Y_val   = data['Y_val'].T.astype(np.float32).reshape(-1, 1)
X_test  = data['X_test'].T.astype(np.float32)
Y_test  = data['Y_test'].T.astype(np.float32).reshape(-1, 1)

# If feature count doesn't match data, take first FEATURE_COUNT columns
if X_train.shape[1] < FEATURE_COUNT:
    raise ValueError(f"Data has only {X_train.shape[1]} features, but FEATURE_COUNT={FEATURE_COUNT}")
elif X_train.shape[1] != FEATURE_COUNT:
    # Truncate / select first N features
    X_train = X_train[:, :FEATURE_COUNT]
    X_val   = X_val[:, :FEATURE_COUNT]
    X_test  = X_test[:, :FEATURE_COUNT]

print(f"Training set: {X_train.shape}, Validation set: {X_val.shape}, Test set: {X_test.shape}")


def build_model(input_dim,
                hidden1=32,
                hidden2=32,
                l2_lambda=1e-4,
                learning_rate=1e-3):
    he_init = initializers.he_normal()
    l2_reg = regularizers.l2(l2_lambda)

    inputs = layers.Input(shape=(input_dim,), name="input")
    x = layers.Dense(hidden1,
                     kernel_initializer=he_init,
                     kernel_regularizer=l2_reg,
                     use_bias=True,
                     name="dense_hidden_1")(inputs)
    x = layers.LeakyReLU(alpha=0.1)(x)

    x = layers.Dense(hidden2,
                     kernel_initializer=he_init,
                     kernel_regularizer=l2_reg,
                     use_bias=True,
                     name="dense_hidden_2")(x)
    x = layers.LeakyReLU(alpha=0.1)(x)

    outputs = layers.Dense(1,
                           activation='sigmoid',
                           kernel_initializer=he_init,
                           kernel_regularizer=l2_reg,
                           name="output")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="2hidden_leakyrelu_model")

    opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_model(input_dim=FEATURE_COUNT,
                    hidden1=32,
                    hidden2=32,
                    l2_lambda=L2_LAMBDA,
                    learning_rate=LEARNING_RATE)

model.summary()

# ----------------------------
# Training
# ----------------------------
early = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early],
    verbose=2
)

# Loss and val_loss arrays from training history
costs = history.history['loss']
validation_costs = history.history.get('val_loss', [None]*len(costs))
epochs_list = list(range(1, len(costs)+1))

# ----------------------------
# Test / Evaluation
# ----------------------------
test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)
print(f"\nTest Loss: {test_loss:.6f} | Test Accuracy: {test_acc:.4f}")

# Predictions (probability)
preds_proba = model.predict(X_test).flatten()  # [0,1]
preds_labels = (preds_proba >= 0.5).astype(int)
true_labels = Y_test.flatten().astype(int)

# Confidence in percent
confidences_pct = preds_proba * 100.0

# DataFrame results
df_results = pd.DataFrame({
    'Prediction Confidence (%)': confidences_pct,
    'True Label': true_labels,
    'Predicted Label': preds_labels
})
df_results['Result'] = np.where(df_results['True Label'] == df_results['Predicted Label'], 'CORRECT', 'WRONG')


In [ ]:
# ----------------------------
# Cost plot
# ----------------------------
plt.figure(figsize=(10, 6))
plt.plot(epochs_list, costs, label="Training Cost")
plt.plot(epochs_list, validation_costs, label="Validation Cost")
plt.title(f"Cost Reduction (LR: {LEARNING_RATE * 5}, Hidden: 32-32)")
plt.xlabel("Epoch Count")
plt.ylabel("Cost (J)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(cost_plot_path)
print(f"Cost plot saved: {cost_plot_path}")
plt.show()

print("\nCreating Graph 1 (Correct vs Wrong)...")
plt.figure(figsize=(12, 7))
bins_range = np.arange(50, 101, 5) # 50, 55, 60... 100
sns.histplot(df_results[df_results['Result'] == 'CORRECT']['Prediction Confidence (%)'],
             bins=bins_range, kde=False, color='green', alpha=0.6, label='Correct')
sns.histplot(df_results[df_results['Result'] == 'WRONG']['Prediction Confidence (%)'],
             bins=bins_range, kde=False, color='red', alpha=0.6, label='Wrong')
plt.title('Prediction Confidence Distribution for Correct/Wrong (Test Set)')
plt.xlabel('Model Confidence in Prediction (%)')
plt.ylabel('Number of Matches')
plt.legend()

# Info box (top right corner)
ax = plt.gca()
info_text = (
    f"Data: {data_path}\n"
    f"Model: 2-hidden NN (LeakyReLU)\n"
    f"Hidden Neurons=32-32\n"
    f"LR={LEARNING_RATE * 5}\n"
    f"Epochs Ran={len(epochs_list)}\n"
    f"Test Acc={test_acc:.4f}"
)
ax.text(
    0.99, 0.99, info_text,
    transform=ax.transAxes, ha='right', va='top', fontsize=10,
    bbox=dict(boxstyle='round', fc='white', ec='#dddddd', alpha=0.85)
)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(hist_out_path)
print(f'Confidence Distribution (Correct vs Wrong) saved: {hist_out_path}')
plt.show()

# ----------------------------
# Optional: Save df_results
# ----------------------------
results_csv = OUT_DIR / "df_results.csv"
df_results.to_csv(results_csv, index=False)
print(f"Detailed results saved as CSV: {results_csv}")

print(f"\nModel Success Rate: {test_acc * 100:.2f}%")
